In [1]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt


C:\Users\AlokVerma\anaconda3\Lib\site-packages\pandas\core\computation\expressions.py:22: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.10.1' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED


In [2]:
df = pd.read_json(r"C:\Users\AlokVerma\Desktop\ML DATA\quotes.json")

In [3]:
df.head()

,Quote,Author,Tags,Popularity,Category
0,"Don't cry because it's over, smile because it ...",Dr. Seuss,"[attributed-no-source, cry, crying, experience...",0.155666,life
1,"Don't cry because it's over, smile because it ...",Dr. Seuss,"[attributed-no-source, cry, crying, experience...",0.155666,happiness
2,"I'm selfish, impatient and a little insecure. ...",Marilyn Monroe,"[attributed-no-source, best, life, love, mista...",0.129122,love
3,"I'm selfish, impatient and a little insecure. ...",Marilyn Monroe,"[attributed-no-source, best, life, love, mista...",0.129122,life
4,"I'm selfish, impatient and a little insecure. ...",Marilyn Monroe,"[attributed-no-source, best, life, love, mista...",0.129122,truth


In [4]:
quotes  = df["Quote"]

In [5]:
import string

quotes = quotes.str.lower()

translator = str.maketrans("", "", string.punctuation)

quotes = quotes.apply(lambda x: x.translate(translator))

In [5]:
quotes.head()

0    Don't cry because it's over, smile because it ...
1    Don't cry because it's over, smile because it ...
2    I'm selfish, impatient and a little insecure. ...
3    I'm selfish, impatient and a little insecure. ...
4    I'm selfish, impatient and a little insecure. ...
Name: Quote, dtype: str

In [6]:
from tensorflow.keras.preprocessing.text import Tokenizer


In [7]:
vocab_size = 100

In [8]:
tokenizer = Tokenizer(num_words = vocab_size)


In [9]:
tokenizer.fit_on_texts(quotes)

In [10]:
word_index = tokenizer.word_index

In [11]:
print(word_index)

{'the': 1, 'to': 2, 'and': 3, 'a': 4, 'of': 5, 'you': 6, 'is': 7, 'i': 8, 'in': 9, 'it': 10, 'that': 11, 'be': 12, 'for': 13, 'not': 14, 'are': 15, 'love': 16, 'your': 17, 'my': 18, 'we': 19, 'but': 20, 'with': 21, 'have': 22, 'life': 23, 'if': 24, 'what': 25, 'as': 26, 'all': 27, 'me': 28, 'can': 29, 'when': 30, 'on': 31, 'or': 32, 'one': 33, 'was': 34, 'like': 35, 'will': 36, 'do': 37, 'they': 38, 'he': 39, 'no': 40, 'who': 41, 'so': 42, 'people': 43, 'there': 44, 'at': 45, 'our': 46, 'an': 47, 'this': 48, 'only': 49, 'by': 50, 'know': 51, 'from': 52, 'more': 53, 'just': 54, 'about': 55, "it's": 56, "don't": 57, 'never': 58, 'than': 59, 'because': 60, 'them': 61, 'his': 62, 'god': 63, 'world': 64, 'how': 65, 'has': 66, 'make': 67, 'up': 68, 'time': 69, 'think': 70, 'out': 71, 'things': 72, 'us': 73, 'want': 74, 'hope': 75, 'always': 76, 'her': 77, 'good': 78, 'would': 79, 'way': 80, 'she': 81, 'their': 82, 'which': 83, 'get': 84, 'man': 85, 'then': 86, 'had': 87, 'into': 88, '10w': 8

In [12]:
list(word_index.items())[:10]

[('the', 1),
 ('to', 2),
 ('and', 3),
 ('a', 4),
 ('of', 5),
 ('you', 6),
 ('is', 7),
 ('i', 8),
 ('in', 9),
 ('it', 10)]

In [13]:
sequence = tokenizer.texts_to_sequences(quotes)

In [14]:
quotes[0]

"Don't cry because it's over, smile because it happened."

In [15]:
sequence[0]

[57, 60, 56, 60, 10]

In [16]:
X = []
y = []

for seq in sequence:
    for i in range(1, len(seq)):
        input_seq = seq[:i]      # input
        output_seq = seq[i]      # next word (target)

        X.append(input_seq)
        y.append(output_seq)

In [17]:
max_len = max(len(X) for i in X)

In [18]:
print(max_len)

610034


In [19]:
from tensorflow.keras.preprocessing.sequence import pad_sequences

X_padded = pad_sequences(X, padding='pre')

In [20]:
y = np.array(y)

In [21]:
print(y)

[60 56 60 ... 69  7  5]


In [22]:
from tensorflow.keras.utils import to_categorical

y_one_hot = to_categorical(y,num_classes = vocab_size)

In [23]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense,SimpleRNN

In [24]:
model = Sequential()

model.add(Embedding(input_dim=20000, output_dim=128, input_length=max_len))
model.add(LSTM(128))
model.add(Dense(20000, activation='softmax'))


C:\Users\AlokVerma\anaconda3\Lib\site-packages\keras\src\layers\core\embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


In [25]:
model.compile(
    loss='categorical_crossentropy',   # ✅ no one-hot
    optimizer='adam',
    metrics=['accuracy']
)

In [26]:
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ embedding (Embedding)                │ ?                           │     0 (unbuilt) │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ lstm (LSTM)                          │ ?                           │     0 (unbuilt) │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense (Dense)                        │ ?                           │     0 (unbuilt) │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [27]:
model.build(input_shape=(None, max_len))
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ embedding (Embedding)                │ (None, 610034, 128)         │       2,560,000 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ lstm (LSTM)                          │ (None, 128)                 │         131,584 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense (Dense)                        │ (None, 20000)               │       2,580,000 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 5,271,584 (20.11 MB)

 Trainable params: 5,271,584 (20.11 MB)

 Non-trainable params: 0 (0.00 B)

In [28]:
epochs = 100
batch_size = 128

In [29]:
model.fit(X_padded,y_one_hot,epochs=epochs,batch_size=batch_size, validation_split=0.1)

Epoch 1/100


ValueError: Arguments `target` and `output` must have the same shape. Received: target.shape=(None, 100), output.shape=(None, 20000)